In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

## Lab 12 - Permutation - Example Code (one continuous, one discrete (binary) variable)

### 1. Data Exploration

In [ ]:
#load dataset
tbl_all = Table.read_table('https://raw.githubusercontent.com/lujiec2020/UMBC-IS296-Fall2022/main/data/baby.csv')
tbl_all

In [ ]:
#select only relevant columns
tbl = tbl_all.select('Maternal Smoker', 'Birth Weight')
tbl

In [ ]:
#summary - count of each group
tbl.group('Maternal Smoker')

In [ ]:
#summary - mean value of each group
tbl.group('Maternal Smoker', np.average).barh('Maternal Smoker')

In [ ]:
#summary - overlaid histogram
tbl.hist('Birth Weight', group='Maternal Smoker')

### 2. Permuation Test

In [ ]:
#step1: determine test statistics 
means_table = tbl.group('Maternal Smoker', np.average) # update
means_table

In [ ]:
means = means_table.column(1)
observed_difference = means.item(1) - means.item(0)
observed_difference

In [ ]:
#function to calcuate difference of means
def difference_of_means(table, var1, var2):
    """
    table: name of table, 
    var1: column label of numerical variable, 
    var2: column label of group-label variable, a discrete variable
    Returns: Difference of means of the two groups"""
    #table with the two relevant columns
    reduced = table.select(var1, var2)  
    # table containing group means
    means_table = reduced.group(var2, np.average)
    # array of group means
    means = means_table.column(1)
    return means.item(1) - means.item(0)

In [ ]:
#test function above
difference_of_means(tbl, 'Birth Weight', 'Maternal Smoker') # update

In [ ]:
#step 2: run simulation 
#2a function to run one iteration of simulation
def one_simulated_difference(table, var1, var2):
    """ 
    table: name of table
    var1: column label of numerical variable,
    var2: column label of group-label variable, a discrete avariable
    Returns: Difference of means of the two groups after shuffling labels"""
    # array of shuffled labels
    shuffled_labels = table.sample(with_replacement = False).column(var2)
    # table of numerical variable and shuffled labels
    shuffled_table = table.select(var1).with_column('Shuffled Label', shuffled_labels)
    return difference_of_means(shuffled_table, var1, 'Shuffled Label')  

In [ ]:
#test the function above
one_simulated_difference(tbl, 'Birth Weight', 'Maternal Smoker') # update

In [ ]:
#2b: function to run multiple iterations of simulation
differences = make_array()
for i in np.arange(100):
    new_difference = one_simulated_difference(tbl, 'Birth Weight', 'Maternal Smoker') # update
    differences = np.append(differences, new_difference)
differences

In [ ]:
# step 3: visualize and interprete results 
Table().with_column('Difference Between Group Means', differences).hist()
print('Observed Difference:', observed_difference)
print('p-value=',np.mean(differences<observed_difference))
plots.title('Difference of means Under the Null Hypothesis');